In [1]:
from glob import glob
from tqdm import tqdm
from datetime import datetime
import netCDF4 as nc
import numpy as np
import pandas as pd
import os
import datetime as dt
import xarray as xr
import matplotlib.dates as mdate
import glob
from sklearn.linear_model import LinearRegression

In [17]:
# function to obtain correlation during night time measurements
def get_linear_reg_coef(df):
    x = df["aLH676"].values.reshape(-1, 1)
    y = df["chla"].values.reshape(-1, 1)
    try:
        model = LinearRegression()
        model.fit(x, y)
    except LinAlgError as e:
        print(e)
        print(f"Skipping file.")
        return
    return pd.Series({"gradient": model.coef_.item(), "intercept": model.intercept_.item()})

In [18]:
# Loading netcdf files and applying get_linear_reg_coef function to all measurements

path = r"C:\MA_data\insitu\chla\thetis_timegrid_nc"
files = glob.glob(os.path.join (path, "*.nc"))

df_total = pd.DataFrame()

In [20]:
for file in files:
    print(file)
    # converting netcdf files to dataframes and formatting for easier processing
    data = xr.open_dataset(file)
    
    required_vars = ["par", "chla", "aLH676"]
    missing_vars = [v for v in required_vars if v not in data.data_vars]
    
    if missing_vars:
        print(f"Skipping file {file} (missing variables: {missing_vars})")
        continue
        
    df = data.to_dataframe()
    df = df.reset_index()
    df_chl = df[["depth", "time", "par", "chla", "aLH676"]]
    df_chl = df_chl.dropna()
    df_chl = df_chl.drop_duplicates(["time", "par"], keep="first").sort_values(["time","depth"])
    df_chl = df_chl.rename(columns={"time":"datetime"})
    df_chl["date"] = pd.to_datetime(df_chl["datetime"]).dt.date
    df_chl["time"] = pd.to_datetime(df_chl["datetime"]).dt.time
    
    #print(df_chl)
    
    # isolating night time measurements using PAR and depth of < 3 m to ensure that the low PAR values at depth during day time are not included
    df_chl_night = df_chl[(df_chl["depth"] < 3) & (df_chl["par"] < 50)]
    df_chl_night = df_chl.merge(df_chl_night, on=["datetime"])
    df_chl_night = df_chl_night.drop(columns=["depth_y","par_y","chla_y","aLH676_y","date_y","time_y"])
    df_chl_night = df_chl_night.rename(columns={"depth_x":"depth","par_x":"par","chla_x":"chla","aLH676_x":"aLH676","date_x":"date","time_x":"time"})
    
    #calculating linear regression to night time measurements
    df_chl_coef = df_chl_night.groupby(["date", "time"])[["chla", "aLH676"]].apply(lambda group: get_linear_reg_coef(group))
    df_chl_coef = df_chl_coef.reset_index()
    df_chl_coef = df_chl_coef.drop_duplicates(["date"], keep="last")
    
    #applying correlation from night time measurements to all measurements
    df_chl = df_chl.merge(df_chl_coef, on=["date"])
    if "gradient" in df_chl.columns:
        df_chl["chla_corr"] = df_chl["gradient"] * df_chl["aLH676"] + df_chl["intercept"]
    else:
        print(f"Skipping file {file} (regression failed)")
    df_chl = df_chl.drop(columns=["time_y"])
    df_chl = df_chl.rename(columns={"time_x":"time"})
    print(df_chl)
    
    # concatenate
    df_total = pd.concat([df_total, df_chl], ignore_index=True)
    

C:\MA_data\insitu\chla\thetis_timegrid_nc\L2_THETIS_GRID_20181018_20181028.nc
       depth                      datetime         par      chla    aLH676  \
0        1.3 2018-10-18 11:17:46.542000128  540.271584  2.449152  0.076602   
1        1.4 2018-10-18 11:17:46.542000128  535.047813  2.443349  0.050994   
2        1.5 2018-10-18 11:17:46.542000128  529.824042  2.437545  0.048589   
3        1.6 2018-10-18 11:17:46.542000128  523.122876  2.453820  0.047874   
4        1.7 2018-10-18 11:17:46.542000128  511.260297  2.470608  0.050307   
...      ...                           ...         ...       ...       ...   
15385   16.7 2018-10-27 21:00:11.698999808    0.040365  1.598496  0.036972   
15386   17.6 2018-10-27 21:00:11.698999808    0.040377  1.484535  0.038203   
15387   17.7 2018-10-27 21:00:11.698999808    0.040381  1.465938  0.036200   
15388   44.0 2018-10-27 21:00:11.698999808    0.040359  0.714037  0.011722   
15389   44.1 2018-10-27 21:00:11.698999808    0.040332  0.741217

In [21]:
print(df_total)

         depth                      datetime         par      chla    aLH676  \
0          1.3 2018-10-18 11:17:46.542000128  540.271584  2.449152  0.076602   
1          1.4 2018-10-18 11:17:46.542000128  535.047813  2.443349  0.050994   
2          1.5 2018-10-18 11:17:46.542000128  529.824042  2.437545  0.048589   
3          1.6 2018-10-18 11:17:46.542000128  523.122876  2.453820  0.047874   
4          1.7 2018-10-18 11:17:46.542000128  511.260297  2.470608  0.050307   
...        ...                           ...         ...       ...       ...   
1066831   48.2 2025-02-18 23:00:10.682999808    0.040343  1.246697  0.017753   
1066832   48.3 2025-02-18 23:00:10.682999808    0.040340  1.248637  0.017748   
1066833   48.4 2025-02-18 23:00:10.682999808    0.040337  1.250578  0.017317   
1066834   48.5 2025-02-18 23:00:10.682999808    0.040334  1.252519  0.016886   
1066835   48.6 2025-02-18 23:00:10.682999808    0.040332  1.254459  0.017207   

               date             time   

In [24]:
df_total.to_csv(r"C:\MA_data\insitu\chla\thetis_corrected_remika\df_thetis_chla_cor.csv", sep=";", encoding="utf-8-sig", index=False)